In [ ]:
#Transform Matrices tentative
bottom_row=np.array([0,0,0,1])
rot45y = np.array([[np.cos(np.pi/4),0,np.sin(np.pi/4)],
                   [0,1,0],
                   [-np.sin(np.pi/4),0,np.cos(np.pi/4)]])
rot45z = np.array([[np.cos(np.pi/4),-np.sin(np.pi/4),0]
                   ,[np.sin(np.pi/4),np.cos(np.pi/4),0]
                   ,[0,0,1]])

#should give the same format as np.array I'm just trying to make easier to tweak
Tce_standoff=np.vstack([np.hstack([rot45y,np.array([[1],[0],[0.5]])]),[bottom_row]])



In [ ]:
#FeedBackControl achieves the goal of ... 

#Xd is ...
#Xd next is ...
#Kp,Ki are P,I gain matrices for the feedback portion
#states_and_angles is ...
def FeedbackControl2(states_and_angles,current_traj,next_traj, Kp, Ki, time_step):

    #constants 
    l = 0.47/2 #m
    w= 0.3/2  #m
    r = 0.0475 #m

    ##Jacobian
    #Blist from exercise 13.33 in MR textbook
    Blist = np.array([[0, 0, 1, 0, 0.033, 0], [0, -1, 0, -0.5076, 0, 0],[0, -1, 0, -0.3526, 0, 0],[0, -1, 0, -0.2176, 0, 0],[0, 0, 1, 0, 0, 0]]).T
    #Arm Jacobian
    J_arm = mr.JacobianBody(Blist, states_and_angles[3:])

    #M0e is the transformation matrix of the end effector position at zero configuration (Ex 13.33)
    M0e = np.array([[1, 0, 0, 0.033],[0, 1, 0, 0],[0, 0, 1, 0.6546],[0, 0, 0, 1]])
    #Arm's end effector position  
    T0e = mr.FKinBody(M0e, Blist, states_and_angles[3:])
    #Chassis to Arm Transformation matrix (Ex 13.33) 
    Tb0 = np.array([[1, 0, 0, 0.1662], [0, 1, 0, 0], [0, 0, 1, 0.0026], [0, 0, 0, 1]])
    #end effector to chassis 
    Teb = mr.TransInv(Tb0 @ T0e) 
    #new F calculation (6x4 matrix consisting of F in wz, vx, vy directions only)
    F= (r/4)*np.array([[-1/(l+w), 1/(l+w), 1/(l+w), -1/(l+w)], [1,1,1,1], [-1,1,-1,1]])
    F_new = np.zeros((6,4))
    F_new[2:5,:] = F 

    #Base Jacobian
    J_base = mr.Adjoint(Teb) @ F_new 

    #full Jacobian
    Je = np.hstack((J_base, J_arm))
    print("Je = ", Je)

    #update X 
    Tsb = np.array([[np.cos(states_and_angles[0]), -np.sin(states_and_angles[0]), 0, states_and_angles[1]], [np.sin(states_and_angles[0]), np.cos(states_and_angles[0]), 0, states_and_angles[2]], [0,0,1,0.0963], [0,0,0,1]])
    X = Tsb @ Tb0 @ T0e
    print("X=", X)

    Tsb_ideal_current = np.array([[np.cos(current_traj[0]), -np.sin(current_traj[0]), 0, current_traj[1]], [np.sin(current_traj[0]), np.cos(current_traj[0]), 0, current_traj[2]], [0,0,1,0.0963], [0,0,0,1]])

    Tsb_ideal_next = np.array([[np.cos(next_traj[0]), -np.sin(next_traj[0]), 0, next_traj[1]], [np.sin(next_traj[0]), np.cos(next_traj[0]), 0, next_traj[2]], [0,0,1,0.0963], [0,0,0,1]])

    Xd=Tsb_ideal_current @ Tb0 @ T0e

    Xd_next=Tsb_ideal_next @ Tb0 @ T0e
    
    #defining global var for integral error 
    #defining global var for integral error 
    global integral_error   

    
    #finding V commanded based on formula in equation 13.37 
    X_err_matrix = mr.MatrixLog6(mr.TransInv(X) @ Xd)
    print("X error",X_err_matrix)
    X_err_twist = mr.se3ToVec(X_err_matrix)
    adjoint_X_Xd = mr.Adjoint(mr.TransInv(X) @ Xd)
    Vd_matrix = (1/time_step)*mr.MatrixLog6(mr.TransInv(Xd) @ Xd_next)
    Vd = mr.se3ToVec(Vd_matrix)
    print("Vd=", Vd)

    #need integral error to be a running total
    integral_error += X_err_twist*time_step 

    V_commanded = (adjoint_X_Xd @ Vd) + (Kp @ X_err_twist) + (Ki @ (integral_error))
    print("V_commanded = ", V_commanded)

    #finding commanded wheel and arm joint speeds 
    total_speeds = np.linalg.pinv(Je) @ V_commanded
    wheel_speeds = total_speeds[0:4]
    arm_joint_speeds = total_speeds[4:]
    print("total_speeds =", total_speeds)
    return wheel_speeds, arm_joint_speeds, V_commanded, X_err_twist, total_speeds

In [ ]:
def cut_off_wheel_speeds(kinematic_prediction):
    states_and_angles=kinematic_prediction[0:8]
    return states_and_angles

In [ ]:
magnitude=100
dt=0.01
kp=np.zeros(6)
ki=np.zeros(6)
full_predictions=[]
vel_command=np.array([1,0,0,0,0,1,1,1,1])
#feed-forward test
#result is the output of trajectory_generator
for i,next_i in zip(result,result[1:]):
    kinematic_prediction=NextState(i,vel_command,dt,magnitude)
    states_and_angles=cut_off_wheel_speeds(kinematic_prediction)
    current_ideal=np.array(cut_off_wheel_speeds(i))
    next_ideal=cut_off_wheel_speeds(next_i)
    print(states_and_angles)
    
    wheel_speeds, arm_joint_speeds, V_commanded, X_err_twist, total_speeds=FeedbackControl2(states_and_angles,current_ideal,next_ideal,kp,ki,dt)
    
    vel_command=total_speeds
    print("Vel_command",vel_command)
    
    #full_predictions.append(kinematic_prediction)

#np.savetxt("feedforward_test.csv",full_predictions,delimiter=",")